# Project-H Quickstart

**~5 min on CPU, ~2 min on T4 GPU**

Train a joystick action head on top of a frozen SmolVLM to navigate a visual target task. Uses `VisionBridge` for a skip connection from the vision encoder — this is what makes it converge fast.

## 1. Install + Setup

In [ ]:
!pip install -q git+https://github.com/jerod92/project-h.git@claude/vla-robotic-hands-platform-kGiza

import torch

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

## 2. Environment Preview

In [ ]:
import matplotlib.pyplot as plt
from project_h import TargetNavEnvironment, run_expert_baseline

env = TargetNavEnvironment(width=224, height=224, max_steps=80)
obs = env.reset(seed=0)

plt.figure(figsize=(3, 3))
plt.imshow(obs)
plt.axis("off")
plt.title("TargetNav: move red circle to blue X")
plt.show()

expert_stats = run_expert_baseline(env, n_episodes=10)
print(f"Expert success rate: {expert_stats['success_rate']:.1%}")

## 3. Load VLM

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq
from project_h import VLAGraft

model_id  = "HuggingFaceTB/SmolVLM-256M-Instruct"
processor = AutoProcessor.from_pretrained(model_id)
vlm       = AutoModelForVision2Seq.from_pretrained(model_id, torch_dtype=torch.float16, low_cpu_mem_usage=True).to(device)

for p in vlm.parameters():
    p.requires_grad = False

vision_dim = VLAGraft.detect_vision_dim(vlm)
print(f"VLM loaded and frozen. Vision dim: {vision_dim}")

## 4. Build Graft

`VisionBridge` adds a skip connection from the vision encoder directly into the action head, bypassing the LLM bottleneck. Without it convergence is ~30% success; with it ~100%.

In [ ]:
from project_h import JoystickAppendage, VisionBridge, VLAGraft, GraftConfig

appendage = JoystickAppendage(hidden_dim=256)
bridged   = VisionBridge(appendage=appendage, vision_dim=vision_dim)
graft     = VLAGraft(vlm=vlm, appendage=bridged, config=GraftConfig())

trainable = sum(p.numel() for p in graft.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")

## 5. Train

In [ ]:
from project_h import CurriculumConfig, TrainingCurriculum, QUICK_CURRICULUM

config = CurriculumConfig(
    bc_steps=400,
    bc_early_stop_loss=0.03,
    rl_steps=50,
    rl_early_stop_success=0.9,
    appendage_lr=3e-4,
    device=device,
    freezing_stages=QUICK_CURRICULUM,
)

history = TrainingCurriculum(graft=graft, processor=processor, env=env, config=config).run()
print("Done.")

## 6. Results

In [ ]:
from project_h import BenchmarkSuite

results = BenchmarkSuite(graft=graft, processor=processor, envs=[env], device=device).run_benchmark(env=env, n_episodes=20)
print(f"Success rate: {results['success_rate']:.1%}")

## 7. GIF

In [ ]:
import os
from project_h import save_rollout_gif
from IPython.display import Image as IPImage

os.makedirs("gifs", exist_ok=True)

save_rollout_gif(
    graft=graft,
    processor=processor,
    env=env,
    path="gifs/quickstart.gif",
    n_steps=80,
    seed=0,
    device=device,
    fps=10,
)

IPImage("gifs/quickstart.gif")